### Import libraries

In [32]:
import os
import json
from pathlib import Path

LOCAL_DATA_ROOT = Path(os.getcwd()).resolve().parent / "data"

import torch
import numpy as np
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


### Set HPs

In [27]:
REPO_ID = "openslr/librispeech_asr"
CONFIG = "clean"
SPLIT = "train.100"
DEFAULT_MODEL = "openai/whisper-large-v3"

### Load dataset

In [ ]:
snapshots = (
    LOCAL_DATA_ROOT /
    "hub" /
    f"datasets--{REPO_ID.replace('/', '--')}" /
    "snapshots"
)
available = sorted(p for p in snapshots.glob("*") if p.is_dir())

assert available, f"no LibriSpeech cache under {snapshots} — run scripts.download_librispeech first"
assert len(available) == 1, f"{len(available)} revisions cached under {snapshots}, expected one"

split_dir = available[0] / CONFIG / SPLIT
shards = sorted(split_dir.glob("*.parquet"))
shards = [str(s) for s in shards]

ds = load_dataset("parquet", data_files=shards, split="train")

### Load Features

In [44]:
audio_feature_dir = (
    LOCAL_DATA_ROOT /
    "features" /
    DEFAULT_MODEL.split("/")[-1] /
    CONFIG /
    SPLIT
)
indexes = sorted(audio_feature_dir.glob("*.json"))

NUM_SAMPLES = 2000
SEED = 0
n_per_shard = NUM_SAMPLES // len(indexes)

rng = np.random.default_rng(SEED)

# Each .npy is a flat (frames, d_model) array holding one shard's utterances back to
# back, so an utterance is the slice feats[offset : offset + length]. Strata are shards:
# draw N_PER_SHARD utterances from each, and materialise only those slices off the mmap.
audio_feats = []
utterances = []
for index_path in indexes:
    utts = json.loads(index_path.read_text())["utts"]
    print(f"Shard has {len(utts)} utterances, sampling {n_per_shard}")
    feats = np.load(index_path.with_suffix(".npy"), mmap_mode="r")
    for i in rng.choice(len(utts), min(n_per_shard, len(utts)), replace=False):
        utt = utts[i]
        audio_feats.append(np.asarray(feats[utt["offset"]:utt["offset"] + utt["length"]]))
        utterances.append(utt["text"])

print(f"Sampled {len(audio_feats)} utterances from {len(indexes)} shards")
print(
    f"{sum(f.shape[0] for f in audio_feats)} frames, "
    f"{sum(f.nbytes for f in audio_feats) / 1e9:.1f} GB in RAM"
)

Shard has 2039 utterances, sampling 142
Shard has 2039 utterances, sampling 142
Shard has 2039 utterances, sampling 142
Shard has 2039 utterances, sampling 142
Shard has 2039 utterances, sampling 142
Shard has 2039 utterances, sampling 142
Shard has 2039 utterances, sampling 142
Shard has 2038 utterances, sampling 142
Shard has 2038 utterances, sampling 142
Shard has 2038 utterances, sampling 142
Shard has 2038 utterances, sampling 142
Shard has 2038 utterances, sampling 142
Shard has 2038 utterances, sampling 142
Shard has 2038 utterances, sampling 142
Sampled 1988 utterances from 14 shards
1266670 frames, 3.2 GB in RAM
